# 🚀 Google Kaggle AI Agents Intensive Capstone Project

**Welcome to my Google Kaggle AI Agents Intensive Capstone Project!**

In this project, I’ve just built and tested a fully functional Google Maps–powered AI assistant that can find places like Italian restaurants near London using live API data.

This project concept leverages Generative AI Agents and Google Maps Platform tools to build an autonomous assistant focused on real-world geospatial tasks, enhancing productivity for users who need to plan, track, and execute location-based activities.

​The system will use the Google Agent Development Kit (ADK) and the Google Maps built-in tool to create an intelligent, location-aware assistant.

# •🗺️ Project: Geo-Planner: The Autonomous Productivity Agent

# •1. Problem Statement

​People often waste significant time manually coordinating location-based tasks, like planning a multi-stop delivery route, finding the most efficient sequence for errands, or creating a travel itinerary that optimizes for time and user preferences (e.g., avoiding traffic, prioritizing specific types of businesses).

​Geo-Planner solves this by building an autonomous, multi-agent system that handles the entire workflow: planning, optimizing, and tracking location-based tasks, minimizing user input and cognitive load.

# •2. System Architecture

​The project will use a multi-agent system for robust, collaborative task completion.
​Agents Used 

## 🤔 Why do Agents need Tools?

# Agent Responsibilities Tools/Capabilities

1️⃣  **Planner Agent** 
Decomposes user goals into location-based tasks. Manages the overall workflow and feedback loop. Google Maps Tool (for distance/route planning), Memory Tool (to store user's favorite locations/preferences).

2️⃣ **Router Agent**
Optimizes the sequence of locations for efficiency (e.g., shortest travel time/distance for a multi-stop trip). Google Maps Tool (Directions/Routes API), LLM Reasoning (for optimization).

3️⃣ **Context Agent**
Retrieves relevant, real-time local information (e.g., business hours, reviews, real-time traffic). Google Maps Tool (Places API, Contextual View), Web Search Tool (for up-to-date context). 

**The Problem**

Without tools, the agent's knowledge is frozen in time — it can't access today's news or your company's inventory. It has no connection to the outside world, so the agent can't take actions.

**The Solution:** Tools are what transform my isolated LLM into a capable agent that can actually help to get things done.

In this notebook, I'll:

- ✅ Turn my Python functions into Agent tools
- ✅ Build an Agent and use it **as a tool** in another agent
- ✅ **Build my first multi-tool agent**
- ✅ Explore the different tool types in ADK

## ⚙️  Setup

 steps below to set up the environment.

### 1.1: Install dependencies

The Kaggle Notebooks environment includes a pre-installed version of the [google-adk](https://google.github.io/adk-docs/) library for Python and its required dependencies.

To install and use ADK in your own Python development environment outside of this course, you can do so by running:

```
pip install google-adk
```

In [2]:
pip install google-adk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 9.6 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0
  Attempting uninstall: cachetools
    Found existing installation: cachetools 6.2.1
    Uninstalling cachetools-6.2.1:
      Successfully uninstalled cachetools-6.2.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 5.29.5 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.

###  Configure Gemini API Key

This notebook uses the [Gemini API](https://ai.google.dev/gemini-api/), which requires an API key.

**1. Get your API key**

 create an [API key in Google AI Studio](https://aistudio.google.com/app/api-keys).

**2. Add the key to Kaggle Secrets**

Next, I will need to add your API key to my Kaggle Notebook as a Kaggle User Secret.

1. In the top menu bar of the notebook editor, select `Add-ons` then `Secrets`.
2. Create a new secret with the label `GOOGLE_API_KEY`.
3. Paste my API key into the "Value" field and click "Save".
4. Ensure that the checkbox next to `GOOGLE_API_KEY` is selected so that the secret is attached to the notebook.

**3. Authenticate in the notebook**

Run the cell below to access the `GOOGLE_API_KEY` I just saved and set it as an environment variable for the notebook to use:

In [5]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Setup and authentication complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

✅ Setup and authentication complete.


###  Import ADK components

Now, import the specific components you'll need from the Agent Development Kit and the Generative AI library. This keeps your code organized and ensures we have access to the necessary building blocks.

In [7]:
from google.genai import types

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search, AgentTool, ToolContext
from google.adk.code_executors import BuiltInCodeExecutor

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


###  Helper functions

Helper function that prints the generated Python code and results from the code execution tool:

In [8]:
def show_python_code_and_result(response):
    for i in range(len(response)):
        # Check if the response contains a valid function call result from the code executor
        if (
            (response[i].content.parts)
            and (response[i].content.parts[0])
            and (response[i].content.parts[0].function_response)
            and (response[i].content.parts[0].function_response.response)
        ):
            response_code = response[i].content.parts[0].function_response.response
            if "result" in response_code and response_code["result"] != "```":
                if "tool_code" in response_code["result"]:
                    print(
                        "Generated Python Code >> ",
                        response_code["result"].replace("tool_code", ""),
                    )
                else:
                    print("Generated Python Response >> ", response_code["result"])


print("✅ Helper functions defined.")

✅ Helper functions defined.


###  Configure Retry Options

When working with LLMs, you may encounter transient errors like rate limits or temporary service unavailability. Retry options automatically handle these failures by retrying the request with exponential backoff.

In [ ]:
retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

## 🤖 Section : What are Custom Tools?

**Custom Tools** are tools are used to build own agent using my own code and business logic. Unlike built-in tools that come ready-made with ADK, custom tools give my complete control over functionality.

**When to use Custom Tools?**

Built-in tools like Google Search are powerful, but **every business has unique requirements** that generic tools can't handle. Custom tools let me implement my specific business logic, connect to my systems, and solve domain-specific problems. ADK provides multiple custom tool types to handle these scenarios.

 Building Custom Function Tools
Location Mapping Agent

This agent can locate the location in Google map and give suggestion to user query . The agent has two custom tools and follows the workflow:

1. **Geo Map Tool** - Finds location as per user query
2. **Support Agent** - Suggest name of the places


### 🤔 define a Tool?

**Any Python function can become an agent tool** by following these simple guidelines:

1. Create a Python function
2. Follow the best practices listed below
3. Add MY function to the agent's `tools=[]` list and ADK handles the rest automatically.


#### 🏆 ADK Best Practices in Action

Notice how my tools follow ADK best practices:

**1. Dictionary Returns**: Tools return `{"status": "success", "data": ...}` or `{"status": "error", "error_message": ...}`  
**2. Clear Docstrings**: LLMs use docstrings to understand when and how to use tools  
**3. Type Hints**: Enable ADK to generate proper schemas (`str`, `dict`, etc.)  
**4. Error Handling**: Structured error responses help LLMs handle failures gracefully  

These patterns make my tools reliable and easy for LLMs to use correctly.

👉 Let's see this in action with my first tool:

**1. Google Maps–powered assistant agent using the google.adk framework.  loading environment variables safely, checks for the API key, and initializes an LlmAgent.**
 

###  Install googlemaps python-dotenv




In [121]:
!pip install googlemaps python-dotenv


🧰
# Model Context Protocol


Model Context Protocol (MCP) is an open standard that lets agents use community-built integrations. Instead of writing  own integrations and API clients, just connect to an existing MCP server.


**MCP enables agents to:**


✅ Access live, external data from databases, APIs, and services without custom integration code

✅ Leverage community-built tools with standardized interfaces

✅ Scale capabilities by connecting to multiple specialized servers

###  Using MCP with  Agent

The workflow is simple:

1. Choose an MCP Server and tool
2. Create the MCP Toolset (configure connection)
3. Add it to your agent
4. Run and test the agent

###  How MCP Works

MCP connects agent (the **client**) to external **MCP servers** that provide tools:

- **MCP Server**: Provides specific tools (like image generation, database access)
- **MCP Client**: Your agent that uses those tools
- **All servers work the same way** - standardized interface

**Architecture:**
```
┌──────────────────┐
│    Agent     │
│   (MCP Client)   │
└────────┬─────────┘
         │
         │ Standard MCP Protocol
         │
    ┌────┴────┬────────┬────────┐
    │         │        │        │
    ▼         ▼        ▼        ▼
┌────────┐ ┌─────┐ ┌──────┐ ┌─────┐
│ GitHub │ │Slack│ │ Maps │ │ ... │
│ Server │ │ MCP │ │ MCP  │ │     │
└────────┘ └─────┘ └──────┘ └─────┘

###  setting the API key directly in my script before using it:




In [53]:
import os
from dotenv import load_dotenv

 
os.environ["GOOGLE_MAPS_API_KEY"] = ""



###  MCPToolset configuration




In [12]:
from google.adk.tools.mcp_tool.mcp_toolset import MCPToolset, StdioServerParameters, StdioConnectionParams


 Add MCP tool to agent

 add the mcp_server to the agent's tool array and update the agent's instructions to handle requests to generate tiny images.

In [21]:
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset

maps_toolset = McpToolset(

    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="path/to/google-maps-cli",
            args=["--api-key", google_maps_api_key]
        )
    )
)


###  Install googlemaps




In [26]:
!pip install googlemaps


  Preparing metadata (setup.py) ... done
  Created wheel for googlemaps: filename=googlemaps-4.10.0-py3-none-any.whl size=40714 sha256=a6ea7048b90b815c55d27984ba34728a885ea35729d2c4c880f2797f70a9fb81
  Stored in directory: /root/.cache/pip/wheels/f1/09/77/3cc2f5659cbc62341b30f806aca2b25e6a26c351daa5b1f49a
Successfully built googlemaps


In [28]:
import googlemaps

gmaps = googlemaps.Client(key="")


In [29]:
gmaps = googlemaps.Client(key="")


In [31]:
def find_places_nearby(query, location="London, UK", radius=5000, type=None):
    geocode_result = gmaps.geocode(location)
    if not geocode_result:
        return f"Could not geocode location: {location}"
    
    latlng = geocode_result[0]['geometry']['location']
    places = gmaps.places_nearby(location=(latlng['lat'], latlng['lng']),
                                 radius=radius,
                                 keyword=query,
                                 type=type)
    
    results = places.get('results', [])
    if not results:
        return f"No places found for '{query}' near {location}."
    
    return [
        {
            "name": place["name"],
            "address": place.get("vicinity"),
            "rating": place.get("rating")
        }
        for place in results[:5]
    ]


In [32]:
import requests
import os

# Set my API key
api_key = os.environ.get("") or ""

def find_places_nearby(query, location="London", radius=5000):
    # Step 1: Geocode the location
    geo_url = f"https://maps.googleapis.com/maps/api/geocode/json?address={location}&key={api_key}"
    geo_resp = requests.get(geo_url).json()
    if not geo_resp.get("results"):
        return f"Could not geocode location: {location}"
    
    latlng = geo_resp["results"][0]["geometry"]["location"]

    # Step 2: Use Places API (New)
    places_url = "https://places.googleapis.com/v1/places:searchNearby"
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": api_key,
        "X-Goog-FieldMask": "places.displayName,places.formattedAddress,places.rating"
    }
    payload = {
        "location": {
            "latitude": latlng["lat"],
            "longitude": latlng["lng"]
        },
        "radius": radius,
        "includedTypes": ["restaurant"],
        "keyword": query
    }

    resp = requests.post(places_url, headers=headers, json=payload).json()
    return [
        {
            "name": p["displayName"]["text"],
            "address": p["formattedAddress"],
            "rating": p.get("rating", "N/A")
        }
        for p in resp.get("places", [])
    ]


In [33]:
results = find_places_nearby("Italian restaurant", location="London")
for i, place in enumerate(results, 1):
    print(f"{i}. {place['name']} - {place['address']} (Rating: {place.get('rating', 'N/A')})")


In [34]:
os.environ["GOOGLE_MAPS_API_KEY"] = ""
api_key = os.environ.get("GOOGLE_MAPS_API_KEY")


In [35]:
def maps_assistant_agent(user_query, location="London", radius=5000):
    # Step 1: Geocode
    geo_url = f"https://maps.googleapis.com/maps/api/geocode/json?address={location}&key={api_key}"
    geo_resp = requests.get(geo_url).json()
    latlng = geo_resp["results"][0]["geometry"]["location"]

    # Step 2: Nearby search
    places_url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"
    params = {
        "location": f"{latlng['lat']},{latlng['lng']}",
        "radius": radius,
        "keyword": user_query,
        "type": "restaurant",
        "key": api_key
    }
    resp = requests.get(places_url, params=params).json()
    ...


In [36]:
user_query = "Italian restaurant"


In [47]:
radius = 5000  # or whatever value you want
params = {
    "location": f"{latlng['lat']},{latlng['lng']}",
    "radius": radius,
    "keyword": user_query,
    "type": "restaurant",
    "key": api_key
}


In [49]:
places_url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"
params = {
    "location": f"{latlng['lat']},{latlng['lng']}",
    "radius": radius,
    "keyword": user_query,
    "type": "restaurant",
    "key": api_key
}
resp = requests.get(places_url, params=params).json()


In [42]:
def maps_assistant_agent(user_query, location="London", radius=5000):
    geo_url = f"https://maps.googleapis.com/maps/api/geocode/json?address={location}&key={api_key}"
    geo_resp = requests.get(geo_url).json()
    if not geo_resp.get("results"):
        return f"Could not geocode location: {location}"
    
    latlng = geo_resp["results"][0]["geometry"]["location"]

    places_url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"
    params = {
        "location": f"{latlng['lat']},{latlng['lng']}",
        "radius": radius,
        "keyword": user_query,
        "type": "restaurant",
        "key": api_key
    }
    resp = requests.get(places_url, params=params).json()
    places = resp.get("results", [])
    if not places:
        return f"No results found for '{user_query}' near {location}."

    response_lines = []
    for i, place in enumerate(places[:5], 1):
        name = place["name"]
        address = place.get("vicinity", "Address not available")
        rating = place.get("rating", "N/A")
        response_lines.append(f"{i}. {name} — {address} (Rating: {rating})")

    return "\n".join(response_lines)


####  🧪 **Agent Test Script**



**Any Python function can become an agent tool** by following these simple guidelines:

1. Create a Python function
2. Follow the best practices listed below
3. Add MY function to the agent's `tools=[]` list and ADK handles the rest automatically.


#### 🧪 Agent Test Script

Notice how my tools follow ADK best practices:

**1. Initializes the agent**

**2. Accepts a user query**

**3. Call the places API**

**4. Return Formatted Results**

# Why This Project Can Win

​Real-World Problem: Directly addresses a common productivity challenge using real-world data (location).

​Strong Tool Integration: Demonstrates mastery of the Google Maps built-in tool (or custom tools for the Maps API) for grounding and function calling.

​Multi-Agent Orchestration: Showcases a clean, three-agent workflow (Planner, Router, Context) that collaborate to produce an optimized output.

​Practical Autonomy: The system moves beyond simple Q&A to plan and optimize a complex task autonomously based on user intent.


 **Google Maps–powered assistant agent using the google.adk framework.  loading environment variables safely, checks for the API key, and initializes an LlmAgent**
 

In [44]:
import requests

api_key = ""
location = "London"
radius = 5000
user_query = "Italian restaurant"

# Step 1: Geocode location
geo_url = f"https://maps.googleapis.com/maps/api/geocode/json?address={location}&key={api_key}"
geo_resp = requests.get(geo_url).json()
latlng = geo_resp["results"][0]["geometry"]["location"]

# Step 2: Nearby search
places_url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"
params = {
    "location": f"{latlng['lat']},{latlng['lng']}",
    "radius": radius,
    "keyword": user_query,
    "type": "restaurant",
    "key": api_key
}
resp = requests.get(places_url, params=params).json()

# Step 3: Display results
places = resp.get("results", [])
for i, place in enumerate(places[:5], 1):
    name = place["name"]
    address = place.get("vicinity", "Address not available")
    rating = place.get("rating", "N/A")
    print(f"{i}. {name} — {address} (Rating: {rating})")


1. Circolo Popolare — 40-41 Rathbone Pl, London (Rating: 4.8)
2. Amalfi Ristorante - Argyll Street — 25 Argyll St, London (Rating: 4.8)
3. Carlotta — 77-78 Marylebone High St, London (Rating: 4.8)
4. Ave Mario — 15 Henrietta St, London (Rating: 4.8)
5. Prezzo Italian Restaurant London Northumberland Avenue — Grand Bldg, 31-32 Northumberland Ave, London (Rating: 4.3)


 Complete Guide to ADK Tool Types

Now that you've seen tools in action, let's understand the complete ADK toolkit:

It's broadly divided into two categories: **Custom tools** and **Built-in tools**

### **1. Custom Tools**

<img src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day2/custom-tools.png" width="800" alt="Custom Tools">



**Advantage**: Complete control over functionality — you build exactly what your agent needs

#### **Function Tools** ✅ (I've used these!)
- **What**: Python functions converted to agent tools
- **Examples**: `get(places_url, params=params).json()`, `get("vicinity", "Address not available")`
- **Advantage**: Turn any Python function into an agent tool instantly

#### **Long Running Function Tools**
- **What**: Functions for operations that take significant time
- **Examples**: Human-in-the-loop approvals, file processing
- **Advantage**: Agents can start tasks and continue with other work while waiting

#### **Agent Tools** ✅ (You've used these!)
- **What**: Other agents used as tools
- **Examples**: `AgentTool(agent=GeoMap)`
- **Advantage**: Build specialist agents and reuse them across different systems

#### **MCP Tools**
- **What**: Tools from Model Context Protocol servers
- **Examples**: Filesystem access, Google Maps, databases
- **Advantage**: Connect to any MCP-compatible service without custom integration

#### **OpenAPI Tools**
- **What**: Tools automatically generated from API specifications
- **Examples**: REST API endpoints become callable tools
- **Advantage**: No manual coding — just provide an API spec and get working tools

### **2. Built-in Tools**

<img src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day2/built-in-tools.png" width="1200" alt="Built-in Tools">

**What**: Pre-built tools provided by ADK

**Advantage**: No development time — use immediately with zero setup

#### **Gemini Tools** ✅ (I've used these!)
- **What**: Tools that leverage Gemini's capabilities
- **Examples**: `google_search`, `BuiltInCodeExecutor`
- **Advantage**: Reliable, tested tools that work out of the box

#### **Google Cloud Tools** [needs Google Cloud access]
- **What**: Tools for Google Cloud services and enterprise integration
- **Examples**: `BigQueryToolset`, `SpannerToolset`, `APIHubToolset`
- **Advantage**: Enterprise-grade database and API access with built-in security

#### **Third-party Tools**
- **What**: Wrappers for existing tool ecosystems
- **Examples**: Hugging Face, Firecrawl, GitHub Tools
- **Advantage**: Reuse existing tool investments — no need to rebuild what already exists

<div align="center">
  <table>
    <tr>
      <th style="text-align:center">Agents Intensive Capstone Project</th>
    </tr>
    <tr>
      <td style="text-align:center"><a href="https://www.linkedin.com/in/vijayarajan-v-3a81271a/"> V.Vijaya Jothi</a></td>
    </tr>
  </table>
</div>